# IRI-2016 Kaggle Verification (Diagnostic Only)

**Purpose.** Verify that the exact published `iricore` release the project selected (D-45, `governance/CHANGE_RECORD_2026-09-19_scientific_decisions.md` and `_p2.md`) actually installs and runs on Kaggle's Linux/CPU environment, with a single permitted smoke-test call at a real, approved station coordinate and a non-December synthetic timestamp -- and nothing more.

**This notebook does NOT:**
- read, download, or reference any GNSS/VTEC target data, ICTP/Madrigal files, or anything under `evidence/locked_test_restricted/`;
- run a full-year or multi-station benchmark;
- register a producer artifact, call `write_release`, or write to `permitted_producers`;
- pass G-04, certify any feature leakage-free, or constitute a scientific result.

**Everything this notebook produces is a diagnostic verification bundle** — an installation/runtime report, not a producer release or a registered benchmark artifact.

**Boundaries respected:** CPU only; no GPU requested or required; no Conda (Kaggle's own Python + a `venv` only); no credentials of any kind are embedded — this notebook reads only public PyPI package files by content hash; no access to any Windows filesystem (everything reads/writes under `/kaggle/working`).

**What is selected, and why (decided before running, not adjusted afterwards):**
- `iricore==1.8.0` — the MOST RECENT published release that ships a prebuilt **Linux** (`manylinux_2_35_x86_64`) wheel, for **CPython 3.10** (`cp310-cp310-manylinux_2_35_x86_64`). Releases 1.8.1–1.9.0 (checked exhaustively against the PyPI JSON API) publish a macOS-arm64 wheel only — no Linux wheel exists at any newer version, so this is a deliberate, verified choice, not a default or an assumption that master equals a release.
- This is **not** the `master` branch; it is the immutable PyPI release archive, pinned by its own published SHA-256 (checked below, before install).
- If Kaggle's own Python is not exactly 3.10, this notebook creates an **isolated `venv`** at Python 3.10 (via `apt-get install python3.10` if the kernel image does not already have it) specifically so the pinned `numpy==1.26.4` (matching the project's own governed pin) and the other exact versions below are never imposed on Kaggle's own base environment. **If Python 3.10 cannot be obtained, this notebook stops with a precise diagnosis — it never silently falls back to a mismatched wheel or an unpinned install.**

In [ ]:
import datetime as dt
import hashlib
import json
import os
import platform
import shutil
import subprocess
import sys
import textwrap
import zipfile
from pathlib import Path

BUNDLE_DIR = Path('/kaggle/working/iri_verification_bundle')
BUNDLE_DIR.mkdir(parents=True, exist_ok=True)
VENV_DIR = Path('/kaggle/working/iri_venv')

report = {
    'notebook': 'kaggle_iri2016_verification.ipynb',
    'purpose': 'diagnostic installation/runtime verification only -- NOT a producer release or registered benchmark artifact',
    'generated_at_utc': dt.datetime.now(dt.timezone.utc).isoformat(),
    'boundaries': {
        'gnss_vtec_target_data_accessed': False,
        'full_year_benchmark_run': False,
        'producer_artifact_registered': False,
        'g04_passed': False,
    },
}


def run(cmd, *, timeout=900, check=False, env=None):
    """Capture a command's real stdout/stderr/exit code -- never inferred. A timeout is
    itself a recorded outcome (returncode None, timed_out True, partial output kept),
    never an uncaught exception that would lose the report."""
    print('$', ' '.join(cmd) if isinstance(cmd, list) else cmd)
    merged_env = dict(os.environ, **(env or {}))
    entry = {'cmd': cmd if isinstance(cmd, str) else ' '.join(cmd), 'timed_out': False}
    try:
        proc = subprocess.run(
            cmd, shell=isinstance(cmd, str), capture_output=True, text=True,
            timeout=timeout, env=merged_env,
        )
        out, err, rc = proc.stdout or '', proc.stderr or '', proc.returncode
    except subprocess.TimeoutExpired as exc:
        def _s(b):
            return b.decode('utf-8', 'replace') if isinstance(b, bytes) else (b or '')
        out, err, rc = _s(exc.stdout), _s(exc.stderr), None
        entry['timed_out'] = True
        entry['timeout_seconds'] = timeout
        print(f'--- TIMED OUT after {timeout}s (recorded, not raised) ---')
    entry.update({'returncode': rc, 'stdout_tail': out[-4000:], 'stderr_tail': err[-4000:]})
    print(out[-2000:])
    if err:
        print('--- stderr (tail) ---')
        print(err[-2000:])
    if check and rc != 0:
        raise RuntimeError(f'command failed (exit {rc}): {entry["cmd"]}')
    return entry


def save_and_stop(reason):
    """Write whatever the report holds so far, zip the bundle, then raise -- a clear
    stop with a precise diagnosis, never a silent fallback to something unverified."""
    report['ok'] = False
    report['stopped_reason'] = reason
    write_bundle()
    raise RuntimeError(f'STOPPED: {reason}')


def write_bundle():
    report_path = BUNDLE_DIR / 'verification_report.json'
    report_path.write_text(json.dumps(report, indent=2, default=str), encoding='utf-8')
    zip_path = Path('/kaggle/working/iri_verification_bundle.zip')
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for p in BUNDLE_DIR.rglob('*'):
            if p.is_file():
                zf.write(p, p.relative_to(BUNDLE_DIR))
    print('Bundle written:', zip_path)
    return zip_path

## Step 1 — Inspect Kaggle's ACTUAL Python/platform before installing anything

Nothing below assumes this matches the project's governed local Python 3.11.16.

In [ ]:
runtime = {
    'python_version': sys.version,
    'python_version_info': list(sys.version_info),
    'python_executable': sys.executable,
    'platform_platform': platform.platform(),
    'platform_machine': platform.machine(),
    'platform_system': platform.system(),
    'in_kaggle': Path('/kaggle').is_dir(),
}
report['runtime'] = runtime
print(json.dumps(runtime, indent=2))

## Step 2 — Decide the environment strategy

`iricore==1.8.0`'s only Linux wheel targets CPython **3.10** exactly (`cp310-cp310-manylinux_2_35_x86_64`). This step decides, from the ACTUAL Python detected above, whether Kaggle's own kernel Python already is 3.10 (then an isolated `venv` is created from it directly, purely to keep the pinned `numpy`/`fortranformat`/`pymap3d` versions from touching Kaggle's own site-packages) or whether a separate `python3.10` binary must be obtained first (via `apt-get`, which needs Kaggle's internet toggle ON). **If neither path is available, this notebook stops here with a precise diagnosis — it does not attempt a from-source Fortran/CMake build.**

In [ ]:
venv_python_source = None
if sys.version_info[:2] == (3, 10):
    venv_python_source = sys.executable
    strategy = 'kernel_python_is_3.10_isolate_via_venv'
else:
    which = shutil.which('python3.10')
    if which:
        venv_python_source = which
        strategy = 'found_existing_python3.10_isolate_via_venv'
    else:
        # No system python3.10: Step 3 will obtain a complete CPython 3.10 through uv's
        # managed interpreters (an immutable published python-build-standalone release),
        # and only as a last resort through apt-get.
        strategy = 'no_system_python3.10_use_uv_managed_3.10'

report['environment_strategy'] = {
    'kernel_python_minor': sys.version_info[:2],
    'strategy': strategy,
    'venv_python_source': venv_python_source,
}
print(json.dumps(report['environment_strategy'], indent=2, default=str))

write_bundle()  # partial report on disk before anything is installed

## Step 3 — Create the isolated environment and install the exact pinned, hash-verified dependencies

**How the isolated environment is created (revised after the first Kaggle run).** The first run found `/usr/bin/python3.10` already present on the Kaggle image and `python3.10 -m venv` failed (exit 1): Debian/Ubuntu ship that interpreter WITHOUT the `python3.10-venv` package, so the stdlib `venv` module has no `ensurepip` and cannot seed pip. This notebook therefore uses **`virtualenv`** — installed into the kernel's own Python via its working pip — which bundles its own pip/setuptools wheels and needs nothing from the target interpreter beyond the binary itself. If even that fails, it tries `apt-get install python3.10-venv` and the stdlib `venv` once more; every attempt's real stdout/stderr/exit code is captured into the report before any stop.

**Second revision (after the second Kaggle run).** That run showed the kernel is Python 3.12, that the `virtualenv -p /usr/bin/python3.10` attempt did not yield a working environment, and that `apt-get` then hung past its 300 s timeout — which, because the helper let `TimeoutExpired` escape, crashed the notebook before the report was written. Three changes: (1) `run()` records a timeout as an outcome and never raises; (2) the report bundle is written after EVERY attempt, so a crash can no longer lose the diagnosis; (3) a new second rung — **`uv`-managed CPython 3.10** (`pip install uv`, `uv==0.12.17`, `uv python install cpython-3.10.21`, `uv venv --seed --python cpython-3.10.21`): uv downloads a complete, immutable python-build-standalone release, needing no apt and nothing from the image's own python3.10. `apt-get` (now `DEBIAN_FRONTEND=noninteractive`, 240 s) is the last rung only.

Every package below is pinned to one exact version with its **published PyPI SHA-256** (fetched from `pypi.org/pypi/<pkg>/<ver>/json` ahead of this run, recorded here verbatim) — `pip install --no-deps --require-hashes` refuses to install anything whose downloaded bytes do not match. `numpy==1.26.4` is the project's own governed pin (`requirements.txt`); `fortranformat==2.0.3` and `pymap3d==3.2.0` are `iricore`'s own declared runtime dependencies, pinned to the newest release inside iricore 1.8.0's declared bounds (`fortranformat>=2.0.0,<3.0.0`; `pymap3d[core]>=3.0.1,<4.0.0`, no upper pin issue since 3.2.0 is the latest).

In [ ]:
env_logs = report.setdefault('installation', {}).setdefault('isolated_env_creation', [])
venv_python = str(VENV_DIR / 'bin' / 'python')

def env_ready():
    return Path(venv_python).is_file() and run([venv_python, '-m', 'pip', '--version'])['returncode'] == 0

mechanism = None

def fresh():
    if VENV_DIR.exists():
        shutil.rmtree(VENV_DIR)

def attempt(label, cmd, **kw):
    env_logs.append({'attempt': label, **run(cmd, **kw)})
    write_bundle()  # the diagnosis is on disk after EVERY attempt

# Rung 1: virtualenv against the image's own python3.10 (only if one exists).
if venv_python_source is not None:
    fresh()
    attempt('rung1: pip install virtualenv into kernel python',
            [sys.executable, '-m', 'pip', 'install', '--quiet', '--no-input', 'virtualenv'], timeout=300)
    attempt('rung1: probe the image python3.10 itself',
            [venv_python_source, '-c', 'import sys, sysconfig; print(sys.version); print(sysconfig.get_paths()["stdlib"])'], timeout=60)
    attempt('rung1: virtualenv -p python3.10',
            [sys.executable, '-m', 'virtualenv', '-p', venv_python_source, str(VENV_DIR)], timeout=300)
    if env_ready():
        mechanism = 'virtualenv against the image python3.10'

# Rung 2: a uv-managed CPython 3.10 (complete standalone interpreter, immutable release).
if mechanism is None:
    fresh()
    attempt('rung2: pip install uv into kernel python',
            [sys.executable, '-m', 'pip', 'install', '--quiet', '--no-input', 'uv==0.12.17'], timeout=300)
    uv_env = {'UV_PYTHON_INSTALL_DIR': '/kaggle/working/uv_python', 'UV_CACHE_DIR': '/kaggle/working/uv_cache'}
    attempt('rung2: uv python install cpython-3.10.21',
            [sys.executable, '-m', 'uv', 'python', 'install', 'cpython-3.10.21'], timeout=600, env=uv_env)
    attempt('rung2: uv python list (what uv will use)',
            [sys.executable, '-m', 'uv', 'python', 'list', '--only-installed'], timeout=120, env=uv_env)
    attempt('rung2: uv venv --seed --python 3.10',
            [sys.executable, '-m', 'uv', 'venv', '--seed', '--python', 'cpython-3.10.21', str(VENV_DIR)], timeout=300, env=uv_env)
    if env_ready():
        mechanism = 'uv 0.12.17 managed cpython-3.10.21 (python-build-standalone release) + uv venv --seed'

# Rung 3 (last resort): Debian's python3.10-venv, non-interactive, bounded.
if mechanism is None and venv_python_source is not None:
    fresh()
    attempt('rung3: apt-get install python3.10-venv (noninteractive)',
            ['bash', '-lc', 'export DEBIAN_FRONTEND=noninteractive; apt-get update -qq && apt-get install -y -qq --no-install-recommends python3.10-venv python3.10-distutils'],
            timeout=240)
    attempt('rung3: python3.10 -m venv', [venv_python_source, '-m', 'venv', str(VENV_DIR)], timeout=300)
    if env_ready():
        mechanism = 'stdlib venv after apt-get python3.10-venv'

if mechanism is None:
    summary = chr(10).join(
        f"  - {e['attempt']}: exit={e['returncode']} timed_out={e.get('timed_out')} "
        f"stderr_tail={(e.get('stderr_tail') or e.get('stdout_tail') or '')[-400:].strip()!r}"
        for e in env_logs
    )
    save_and_stop(
        'no isolated Python 3.10 environment could be created: every rung failed (virtualenv against the image python3.10 / uv-managed CPython 3.10 / apt-get + stdlib venv) -- see report["installation"]["isolated_env_creation"] for each attempt: exact command, exit code, stdout and stderr. Per-attempt summary:' + chr(10) + summary
    )

venv_info = run([venv_python, '-c', 'import sys, platform; print(sys.version); print(platform.platform())'])
report['environment_strategy']['venv_python_version'] = venv_info['stdout_tail'].strip()
report['environment_strategy']['isolated_env_mechanism'] = mechanism
write_bundle()
print(json.dumps(report['environment_strategy'], indent=2, default=str))

In [ ]:
# Exact published wheel hashes (fetched from PyPI's JSON API before this run; the
# only Linux/cp310 wheel each package published at the pinned version).
REQUIREMENTS_TXT = textwrap.dedent('''\
    numpy==1.26.4 --hash=sha256:ffa75af20b44f8dba823498024771d5ac50620e6915abac414251bd971b4529f
    fortranformat==2.0.3 --hash=sha256:88c8e7a3eac16c23420e8a1c4b21ddc7108f48e8dcbd2e0da6c8ecc48b051bb2
    pymap3d==3.2.0 --hash=sha256:fccd44f2f6021a95adec19771c603b8dac104eab120d863c463d76b9bc298669
    iricore==1.8.0 --hash=sha256:f452b22316891d87ee766dba266de6a07e4e6008ab515ffed902ea8b5446a874
''')
req_path = BUNDLE_DIR / 'requirements-iri.txt'
req_path.write_text(REQUIREMENTS_TXT, encoding='utf-8')
print(REQUIREMENTS_TXT)

pip_log = run(
    [venv_python, '-m', 'pip', 'install', '--no-deps', '--require-hashes', '-r', str(req_path)],
    timeout=600,
)
report.setdefault('installation', {})['pip_install'] = pip_log
if pip_log['returncode'] != 0:
    save_and_stop(
        'pip install --require-hashes failed inside the isolated venv (see report["installation"]["pip_install"] for the full captured stdout/stderr and the actual exit code); the most likely cause on Kaggle is a manylinux platform tag mismatch (iricore 1.8.0 requires manylinux_2_35, i.e. a fairly recent glibc) -- this is reported exactly, never silently retried with a different version.'
    )

In [ ]:
freeze = run([venv_python, '-m', 'pip', 'freeze'])
report['installation']['pip_freeze'] = freeze['stdout_tail']

## Step 4 — The inner verification script

Written to disk once, then executed inside the venv exactly as-is; both the "Kaggle already had Python 3.10" path and the "isolated venv at Python 3.10" path run this SAME code, so nothing about the verification itself depends on which branch of Step 2 was taken. It does the provenance check, the index-file hashing, the one permitted smoke-test call (repeated once for a repeatability check), and the source-inspection reconciliation, and prints exactly one JSON object.

In [ ]:
INNER_SCRIPT = r'''
"""iri_inner_verify.py -- runs INSIDE the isolated venv (or the kernel env if it already
matched, per the notebook's strategy decision). Does the actual iricore
import/provenance/smoke-test/repeatability/index-hash work and prints ONE JSON object to
stdout. Never touches GNSS/VTEC target data; never runs a full-year benchmark; the
smoke test is a single permitted point evaluation at a real, approved station
coordinate and a synthetic non-December timestamp -- diagnostic only.
"""
import datetime as dt
import hashlib
import importlib.metadata
import json
import math
import re
import sys
import traceback
from pathlib import Path

OUT = {"ok": False}


def fail(stage, exc):
    OUT["ok"] = False
    OUT["failed_stage"] = stage
    OUT["exception"] = "".join(traceback.format_exception_only(type(exc), exc)).strip()
    print(json.dumps(OUT, default=str))
    sys.exit(1)


try:
    OUT["python"] = {
        "version": sys.version,
        "executable": sys.executable,
    }

    # --- 1. import and provenance -----------------------------------------------------
    OUT["_stage"] = "import_iricore"
    import iricore  # noqa: E402

    OUT["iricore_import_path"] = str(Path(iricore.__file__).resolve())
    try:
        OUT["iricore_version_dist"] = importlib.metadata.version("iricore")
    except importlib.metadata.PackageNotFoundError:
        OUT["iricore_version_dist"] = None
    OUT["iricore_version_attr"] = getattr(iricore, "__version__", None)

    from iricore.config import DEFAULT_IRI_VERSION  # noqa: E402

    OUT["installed_default_iri_version"] = DEFAULT_IRI_VERSION
    # Reconciliation vs the earlier source inspection (CR-2026-09-19-SCI-DECISIONS §3):
    # the wrapper's own default was IRI-2020, not IRI-2016 -- this notebook's smoke test
    # explicitly passes version=16 below and never relies on the default. A mismatch here
    # (e.g. an installed release that flipped the default to 16) is reported, not hidden,
    # because it would change what "explicit version=16" is guarding against.
    OUT["reconciliation"] = {
        "expected_default_from_source_inspection": 20,
        "installed_default_matches_expectation": DEFAULT_IRI_VERSION == 20,
    }

    # --- 2. locate and hash the shipped index files (D-45: pin by hash, detect drift) -
    OUT["_stage"] = "locate_and_hash_index_files_before"
    index_dir = Path(iricore.__file__).resolve().parent / "data" / "index"
    apf107 = index_dir / "apf107.dat"
    ig_rz = index_dir / "ig_rz.dat"
    if not apf107.is_file() or not ig_rz.is_file():
        raise RuntimeError(f"expected index files not found under {index_dir}")

    def sha256_of(path: Path) -> str:
        return hashlib.sha256(path.read_bytes()).hexdigest()

    hashes_before = {"apf107.dat": sha256_of(apf107), "ig_rz.dat": sha256_of(ig_rz)}
    OUT["index_files"] = {"directory": str(index_dir), "sha256_before": hashes_before}

    # --- 3. pick a safe, non-December, well-inside-coverage smoke-test date -----------
    # Parsed from the file's OWN records (13I3,3F5.1 fixed layout) rather than assumed,
    # so the choice is correct for whatever release is actually installed.
    OUT["_stage"] = "parse_apf107_coverage_and_pick_smoke_date"
    last_line = None
    with apf107.open("r", encoding="ascii", errors="strict") as fh:
        for line in fh:
            if line.strip():
                last_line = line
    if last_line is None:
        raise RuntimeError("apf107.dat has no data lines")
    m = re.match(r"\s*(\d{2})\s*(\d{1,2})\s*(\d{1,2})", last_line)
    if not m:
        raise RuntimeError(f"could not parse the last apf107.dat line: {last_line!r}")
    yy, mm, dd = (int(x) for x in m.groups())
    last_year = 1900 + yy if yy >= 32 else 2000 + yy
    last_date = dt.date(last_year, mm, dd)
    # 60 days back from the file's own last covered date, walked to a non-December day.
    candidate = last_date - dt.timedelta(days=60)
    while candidate.month == 12:
        candidate -= dt.timedelta(days=30)
    smoke_time = dt.datetime(candidate.year, candidate.month, candidate.day, 12, 0, 0)
    OUT["index_files"]["last_covered_date_in_apf107"] = last_date.isoformat()
    OUT["smoke_test_timestamp_utc"] = smoke_time.isoformat()
    assert smoke_time.month != 12, "smoke-test date must not be in December (project convention)"

    # --- 4. the approved integration settings -----------------------------------------
    # htop = 2000 km is the ONLY altitude-ceiling value TE/Vision freeze (Vision §6.11,
    # "explicit 2000 km altitude ceiling"). hbot and hstep are NOT frozen by the project;
    # this smoke test uses iricore's OWN wrapper defaults for them, disclosed as such --
    # never invented as if they were approved values.
    ARUC_LAT, ARUC_LON = 40.286, 44.086  # D-1's frozen ARUC station coordinate (public
    # station metadata, NOT a GNSS/VTEC target value; no target data is read here)
    HBOT_KM, HTOP_KM, HSTEP_KM = 90.0, 2000.0, 0.5  # HTOP frozen; HBOT/HSTEP = wrapper defaults
    IRI_VERSION = 16  # explicit -- the installed default is 20 (IRI-2020), never relied on
    OUT["integration_settings"] = {
        "lat": ARUC_LAT,
        "lon": ARUC_LON,
        "station": "ARUC (D-1 frozen coordinate; coordinate only, no target data read)",
        "hbot_km": HBOT_KM,
        "htop_km": HTOP_KM,
        "htop_km_is_te_vision_frozen": True,
        "hstep_km": HSTEP_KM,
        "hstep_km_is_frozen": False,
        "hbot_km_is_frozen": False,
        "iri_version_requested": IRI_VERSION,
        "iri_version_is_default": False,
        "output_unit": "TECU (verified from iricore.tec._integrate_ne: sums Ne*step_km, "
        "converts km->m (*1e3) then to TECU (*1e-16))",
    }

    # --- 5. the smoke test itself, called TWICE for repeatability ---------------------
    OUT["_stage"] = "smoke_test_call_1"
    first = iricore.vtec(
        smoke_time, ARUC_LAT, ARUC_LON, hbot=HBOT_KM, htop=HTOP_KM, hstep=HSTEP_KM,
        version=IRI_VERSION,
    )
    OUT["_stage"] = "smoke_test_call_2_repeatability"
    second = iricore.vtec(
        smoke_time, ARUC_LAT, ARUC_LON, hbot=HBOT_KM, htop=HTOP_KM, hstep=HSTEP_KM,
        version=IRI_VERSION,
    )
    first_val = float(first[0] if hasattr(first, "__len__") else first)
    second_val = float(second[0] if hasattr(second, "__len__") else second)
    finite = math.isfinite(first_val) and math.isfinite(second_val)
    physically_plausible = 0.0 <= first_val <= 200.0  # iricore's own internal sanity bound
    repeatable = first_val == second_val
    OUT["smoke_test"] = {
        "call_1_tecu": first_val,
        "call_2_tecu": second_val,
        "finite": finite,
        "physically_plausible_0_to_200_tecu": physically_plausible,
        "repeatable_bit_identical": repeatable,
    }
    if not finite:
        raise RuntimeError(f"non-finite smoke-test output: {first_val!r}, {second_val!r}")
    if not repeatable:
        raise RuntimeError(
            f"repeated identical call produced different output: {first_val!r} != {second_val!r}"
        )
    if not physically_plausible:
        # NOT fatal by itself (iricore only warns at this same bound) -- but it IS a
        # material mismatch worth failing loudly rather than reporting a quiet PASS.
        raise RuntimeError(
            f"smoke-test output {first_val!r} TECU outside iricore's own plausibility "
            f"bound [0, 200]; reported as a material mismatch, not silently accepted"
        )

    # --- 6. index files must be byte-identical after the calls (no silent refresh) ----
    OUT["_stage"] = "hash_index_files_after"
    hashes_after = {"apf107.dat": sha256_of(apf107), "ig_rz.dat": sha256_of(ig_rz)}
    OUT["index_files"]["sha256_after"] = hashes_after
    OUT["index_files"]["unchanged_after_smoke_test"] = hashes_before == hashes_after
    if hashes_before != hashes_after:
        raise RuntimeError(
            f"index file hash changed after the smoke-test calls (silent update "
            f"detected): before={hashes_before} after={hashes_after}"
        )

    OUT["_stage"] = "done"
    OUT["ok"] = True
    print(json.dumps(OUT, default=str))
except Exception as exc:  # noqa: BLE001 -- this script's whole job is to report, never crash silently
    fail(OUT.get("_stage", "unspecified"), exc)
'''
inner_path = BUNDLE_DIR / 'iri_inner_verify.py'
inner_path.write_text(INNER_SCRIPT, encoding='utf-8')
print('Inner script written to', inner_path)


In [ ]:
inner_log = run([venv_python, str(inner_path)], timeout=300)
report['verification_process'] = {
    'returncode': inner_log['returncode'],
    'stdout_tail': inner_log['stdout_tail'],
    'stderr_tail': inner_log['stderr_tail'],
}
try:
    verification = json.loads(inner_log['stdout_tail'].strip().splitlines()[-1])
except Exception as exc:
    save_and_stop(
        f'the inner verification script did not print a parseable JSON object on its last stdout line (exit code {inner_log["returncode"]}); see report["verification_process"] for the full captured output -- parse error: {exc!r}'
    )
report['verification'] = verification
print(json.dumps(verification, indent=2, default=str))
if not verification.get('ok'):
    save_and_stop(
        f'the inner verification FAILED at stage {verification.get("failed_stage")!r}: {verification.get("exception")}; reported exactly, never silently downgraded to a pass'
    )

## Step 5 — Reconciliation against the earlier (pre-Kaggle) source inspection

`CR-2026-09-19-SCI-DECISIONS.md` §3 inspected `iricore`'s Fortran/Python source directly (the `master` branch and the 1.9.0 sdist) and established: the wrapper's `DEFAULT_IRI_VERSION` is IRI-2020 (never IRI-2016) — this notebook always passes `version=16` explicitly and the inner script asserts the installed default is still 20, failing loudly on drift; daily/81-day/365-day F10.7 read from `apf107.dat` are the **adjusted**, same-day, **centered**-mean quantities documented there, never the project's own observed/trailing convention — no override of these is used, matching D-45's adopted disposition (a disclosed retrospective climatological reference).

In [ ]:
assert verification['reconciliation']['installed_default_matches_expectation'], (
    'installed iricore default IRI version drifted from the source inspection -- STOP, '
    'do not silently alter benchmark semantics'
)
assert verification['integration_settings']['iri_version_requested'] == 16
assert verification['integration_settings']['htop_km'] == 2000.0
assert verification['smoke_test']['repeatable_bit_identical']
assert verification['index_files']['unchanged_after_smoke_test']
print('Reconciliation checks passed: no material mismatch against the source inspection.')
report['reconciliation_passed'] = True

## Step 6 — Finalize the diagnostic bundle

Everything above is written to `/kaggle/working/iri_verification_bundle/` and zipped to `/kaggle/working/iri_verification_bundle.zip` — **download that one file** from the Kaggle output pane. It is a diagnostic report only.

In [ ]:
report['ok'] = True
zip_path = write_bundle()
print('DONE. Download this file from the Kaggle output pane:', zip_path)
print(json.dumps({k: v for k, v in report.items() if k not in ('installation',)}, indent=2, default=str)[:4000])